## RFM分群

## 載入 D2 清洗後的 CLEAN.csv

用相對路徑指到 D2 的 `out/` 資料夾(因為 D4 不重複放原始資料)。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'PingFang TC', 'Noto Sans CJK TC']
plt.rcParams['axes.unicode_minus'] = False

BASE = Path().resolve()
# 路徑 fallback:優先讀 D2 清洗結果(學員把 D2 zip 解在 ../D2_資料清洗/ 的話),
# 否則用本目錄附帶的 sample CLEAN.csv(D4 zip 自帶,確保可獨立跑通)
_candidates = [
    BASE / '..' / 'D2_資料清洗' / 'out' / 'A_物流_訂單配送_CLEAN.csv',
    BASE / 'A_物流_訂單配送_CLEAN.csv',
]
ORDERS = next((p for p in _candidates if p.exists()), _candidates[0])
print(f'📂 載入:{ORDERS.parent.name}/{ORDERS.name}')

df = pd.read_csv(ORDERS, encoding='utf-8-sig', parse_dates=['order_date'])
print(f'訂單資料:{len(df)} 筆 × {len(df.columns)} 欄')
print(f'觀察期:{df["order_date"].min().date()} 到 {df["order_date"].max().date()}')
print(f'客戶數:{df["customer_id"].nunique()}')
df.head(3)

## RFM 三把尺

| 維度 | 定義 | 物流問法 |
|---|---|---|
| **R · Recency** | 最近一次交易距今多久 | 「最近多久沒下單?」 |
| **F · Frequency** | 觀察期內交易次數 | 「半年內下了幾次?」 |
| **M · Monetary** | 觀察期內交易金額 | 「半年內總運費多少?」 |

### 為什麼是這三個?

因為它們分別代表 **時間 × 活躍度 × 規模**。單看一個都不夠 — 要組合才完整。

In [4]:
# 觀察期結束日(2025-07-01,留 D2 cleaned 期間 +幾天 buffer)
snapshot = pd.Timestamp('2025-07-01')
print(f'📅 觀察期結束日:{snapshot.date()}')

# RFM 三維計算
rfm = df.groupby(['customer_id', 'customer_name']).agg(
    R=('order_date', lambda x: (snapshot - x.max()).days),
    F=('order_id', 'count'),
    M=('freight_twd', 'sum'),
).reset_index()

# 按 M 降序看一眼
print(rfm.sort_values('M', ascending=False))

📅 觀察期結束日:2025-07-01
  customer_id customer_name  R    F         M
6        C007          統一生鮮  1  115  270698.4
2        C003      水產鮮活 B2B  1  108  245576.3
1        C002      momo 購物網  1  103  232622.3
7        C008    PChome 24h  1  109  222216.5
5        C006        全聯福利中心  2   93  215567.8
4        C005          蝦皮購物  1   95  199303.3
0        C001     7-11 零售連鎖  1   88  199084.4
3        C004         家樂福全聯  1   82  171144.1


## Part A · 五等分(qcut)+ RFM_score

把每個維度切成 5 等(分位數)→ 用 1-5 編碼。

**注意**:R 越小越好(最近),所以分數**反向**(低 R → 高分);F / M 越大越好(正向)。

In [11]:
# R:越小越好 → 反向編碼
rfm['R_score'] = pd.qcut(rfm['R'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1]).astype(int)

# F:越大越好,加 rank(method='first') 處理重複值
rfm['F_score'] = pd.qcut(rfm['F'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

# M:越大越好
rfm['M_score'] = pd.qcut(rfm['M'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

# RFM 三位字串(125 種組合)
rfm['RFM_score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

print(rfm[['customer_id', 'customer_name', 'R', 'F', 'M',
           'R_score', 'F_score', 'M_score', 'RFM_score']]
      .sort_values('M', ascending=False))

  customer_id customer_name  R    F         M  R_score  F_score  M_score  \
6        C007          統一生鮮  1  115  270698.4        2        5        5   
2        C003      水產鮮活 B2B  1  108  245576.3        4        4        5   
1        C002      momo 購物網  1  103  232622.3        5        3        4   
7        C008    PChome 24h  1  109  222216.5        1        5        3   
5        C006        全聯福利中心  2   93  215567.8        1        2        3   
4        C005          蝦皮購物  1   95  199303.3        3        3        2   
0        C001     7-11 零售連鎖  1   88  199084.4        5        1        1   
3        C004         家樂福全聯  1   82  171144.1        3        1        1   

  RFM_score  
6       255  
2       445  
1       534  
7       153  
5       123  
4       332  
0       511  
3       311  


## Part B · 八分群對應 + 行動建議

從 125 種組合濃縮成 8 個典型分群。判斷邏輯:

| 分群 | R | F | M | 策略 emoji |
|---|:---:|:---:|:---:|:---:|
| **VIP(Champions)** | 高 | 高 | 高 | 🛡 鎖 |
| **Loyal** | 中高 | 高 | 中 | 🌱 養 |
| **Big Spenders** | 高 | 低 | 高 | 💸 挖 |
| **New / Promising** | 高 | 低 | 低 | 🌀 引 |
| **Potential Loyalists** | 高 | 中 | 中 | 🌱 扶 |
| **At Risk** | 低 | 中高 | 高 | 🚨 救 |
| **Hibernating** | 低 | 低 | 中 | ⏰ 喚 |
| **Lost** | 低 | 低 | 低 | 👋 放 |

In [12]:
def classify(row):
    R, F, M = row['R_score'], row['F_score'], row['M_score']
    if R >= 4 and F >= 4 and M >= 4:
        return 'VIP (Champions) 🛡 鎖'
    elif R >= 3 and F >= 4 and M >= 3:
        return 'Loyal 🌱 養'
    elif R >= 4 and F <= 2 and M >= 4:
        return 'Big Spenders 💸 挖'
    elif R >= 4 and F <= 2 and M <= 2:
        return 'New/Promising 🌀 引'
    elif R >= 4 and F == 3:
        return 'Potential Loyalists 🌱 扶'
    elif R <= 2 and F >= 3 and M >= 3:
        return 'At Risk 🚨 救'
    elif R <= 2 and F <= 2 and M >= 3:
        return 'Hibernating ⏰ 喚'
    else:
        return 'Lost 👋 放'

rfm['segment'] = rfm.apply(classify, axis=1)

print('📊 分群結果:')
for seg in rfm['segment'].unique():
    members = rfm[rfm['segment'] == seg]['customer_name'].tolist()
    print(f'  {seg}:{", ".join(members)}({len(members)} 個)')

📊 分群結果:
  New/Promising 🌀 引:7-11 零售連鎖(1 個)
  Potential Loyalists 🌱 扶:momo 購物網(1 個)
  VIP (Champions) 🛡 鎖:水產鮮活 B2B(1 個)
  Lost 👋 放:家樂福全聯, 蝦皮購物(2 個)
  Hibernating ⏰ 喚:全聯福利中心(1 個)
  At Risk 🚨 救:統一生鮮, PChome 24h(2 個)


## Part C · Top 20% 名單 + At Risk 警示

為任務 05 業務行動書準備兩條清單:

1. **Top 20%** — 鎖定的 VIP / Big Spenders / Loyal
2. **At Risk** — 即將流失的高價值客戶,要用力救

In [13]:
# Top 20% 客戶(按 M 降序)
n_top = max(int(len(rfm) * 0.2), 1)
top20 = rfm.nlargest(n_top, 'M')[['customer_name', 'segment', 'R', 'F', 'M']]
print('🏆 Top 20% 客戶名單(鎖定)')
print(top20.to_string(index=False))

# At Risk 名單
at_risk = rfm[rfm['segment'].str.contains('At Risk')][['customer_name', 'R', 'F', 'M']]
print('\n🚨 At Risk 名單(救援)')
if len(at_risk):
    print(at_risk.to_string(index=False))
else:
    print('  (這次資料中沒有客戶落入 At Risk 群,但要持續監控 R 分數變動)')

🏆 Top 20% 客戶名單(鎖定)
customer_name     segment  R   F        M
         統一生鮮 At Risk 🚨 救  1 115 270698.4

🚨 At Risk 名單(救援)
customer_name  R   F        M
         統一生鮮  1 115 270698.4
   PChome 24h  1 109 222216.5
